# Experiment: Physical VRAM Profiling (Unprocessed vs. Branch 1 Processed Weights)

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25].mlp.gate_proj`)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Objective:
Physically measure and compare the GPU VRAM footprint between:
1. **Unprocessed Pristine Model**: Standard dense `nn.Linear` layers ($6912 \times 1152$).
2. **Processed Model (Branch 1)**: Structural parameter replacement via `TuckerFactorizedGateProj`, storing only the small Tucker core $\mathcal{S}$, factor matrices $A, B, C$, and uncompressed superweights on the GPU.

### Metrics Measured:
- **Static Parameter VRAM (MB)**: Exact GPU memory occupied by model parameters.
- **Allocated GPU Memory (MB)**: PyTorch `torch.cuda.memory_allocated()`.
- **Reserved GPU Memory (MB)**: PyTorch caching allocator `torch.cuda.memory_reserved()`.
- **Peak Runtime VRAM (MB)**: Maximum memory allocated during live inference (`torch.cuda.max_memory_allocated()`).

In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
import gc
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

In [ ]:
# =====================================================================
# STEP 2: Precise GPU VRAM Measurement Utilities
# =====================================================================
def get_model_param_vram_mb(mod):
    """Calculates exact memory (MB) occupied by model parameters and buffers on CUDA."""
    param_bytes = sum(p.numel() * p.element_size() for p in mod.parameters() if p.is_cuda)
    buffer_bytes = sum(b.numel() * b.element_size() for b in mod.buffers() if b.is_cuda)
    return (param_bytes + buffer_bytes) / (1024 ** 2)

def get_cuda_memory_snapshot():
    """Returns current allocated, reserved, and peak allocated VRAM in MB."""
    if not torch.cuda.is_available():
        return {"allocated_mb": 0.0, "reserved_mb": 0.0, "max_allocated_mb": 0.0}
    return {
        "allocated_mb": round(torch.cuda.memory_allocated() / (1024 ** 2), 2),
        "reserved_mb": round(torch.cuda.memory_reserved() / (1024 ** 2), 2),
        "max_allocated_mb": round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2),
    }

def clear_cuda_cache():
    """Clears unused cached memory from the PyTorch allocator."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

print("VRAM profiling utilities ready.")

In [ ]:
# =====================================================================
# STEP 3: Initialize Model & Tokenizer on GPU (Unprocessed Baseline)
# =====================================================================
clear_cuda_cache()

model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
print(f"Loaded model with {NUM_LAYERS} transformer decoder layers.")

# Save pristine unprocessed weights for all 26 gate_proj layers
W_gate_orig_all = {
    l: model.model.layers[l].mlp.gate_proj.weight.data.clone()
    for l in range(NUM_LAYERS)
}

# Initial Static VRAM Footprint
unprocessed_param_vram = get_model_param_vram_mb(model)
unprocessed_gate_vram = sum(
    model.model.layers[l].mlp.gate_proj.weight.numel() * model.model.layers[l].mlp.gate_proj.weight.element_size()
    for l in range(NUM_LAYERS) if model.model.layers[l].mlp.gate_proj.weight.is_cuda
) / (1024 ** 2)

print(f"Total Model Parameter VRAM:      {unprocessed_param_vram:.2f} MB")
print(f"26 gate_proj Layers Weight VRAM: {unprocessed_gate_vram:.2f} MB")

In [ ]:
# =====================================================================
# STEP 4: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

EVAL_SAMPLE_COUNT = 150
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

## Step 1: Benchmark Unprocessed Model VRAM (Static & Peak Runtime)

We measure the pristine, uncompressed model's GPU memory:
1. **Allocated Memory**: GPU memory held by the loaded weights.
2. **Peak Runtime Memory**: Maximum VRAM reached during GLUE MNLI inference passes.
3. Forward hooks simultaneously record empirical activations across all 26 layers for Branch 1 decomposition.

In [ ]:
# =====================================================================
# STEP 5: Baseline Benchmark & Peak VRAM Measurement
# =====================================================================
clear_cuda_cache()

layer_trajectories = {l: [] for l in range(NUM_LAYERS)}
current_acts = {}

def make_act_hook(layer_idx):
    def hook_fn(module, input_tensor, output_tensor):
        act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        current_acts[layer_idx] = act.detach().cpu()
    return hook_fn

hooks = [
    model.model.layers[l].mlp.act_fn.register_forward_hook(make_act_hook(l))
    for l in range(NUM_LAYERS)
]

baseline_snapshot_before = get_cuda_memory_snapshot()

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Unprocessed Model Inference & Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        for l in range(NUM_LAYERS):
            if l in current_acts and current_acts[l] is not None:
                pooled = current_acts[l].squeeze(0).mean(dim=0).numpy()
                layer_trajectories[l].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in hooks:
    h.remove()

baseline_snapshot_after = get_cuda_memory_snapshot()
acts_matrices = {l: np.stack(layer_trajectories[l]) for l in range(NUM_LAYERS)}

baseline_accuracy = accuracy_score(ground_truth, predictions)
unprocessed_peak_vram = baseline_snapshot_after["max_allocated_mb"]
baseline_peak_vram = unprocessed_peak_vram

print(f"\nUnprocessed Model Benchmark Results:")
print(f"  Accuracy:                {baseline_accuracy * 100:.2f}%")
print(f"  Static Parameter VRAM:   {unprocessed_param_vram:.2f} MB")
print(f"  Allocated VRAM:          {baseline_snapshot_after['allocated_mb']:.2f} MB")
print(f"  Peak Runtime VRAM:       {unprocessed_peak_vram:.2f} MB")

## Step 2: Structural Module Replacement via `TuckerFactorizedGateProj`

To physically reduce GPU VRAM, we define `TuckerFactorizedGateProj`:
Instead of storing the full $6912 \times 1152$ dense matrix ($7.96\text{M}$ parameters / $31.85\text{ MB}$ per layer), the module stores ONLY:
1. **Uncompressed Superweights**: $W_{\text{super}} \in \mathbb{R}^{N_{\text{super}} \times 1152}$
2. **Compact Tucker Factors**: Core $\mathcal{S} \in \mathbb{R}^{R_1 \times R_2 \times R_3}$ and matrices $U^{(1)}, U^{(2)}, U^{(3)}$
3. **Remaining Inactive/Non-Cluster Coordinates**: $W_{\text{remaining}} \in \mathbb{R}^{N_{\text{rem}} \times 1152}$

We replace `model.model.layers[l].mlp.gate_proj` across all 26 layers, delete original dense weights, and clear CUDA cache.

In [ ]:
# =====================================================================
# STEP 6: Define TuckerFactorizedGateProj & Replace All 26 Layers
# =====================================================================
class TuckerFactorizedGateProj(torch.nn.Module):
    """
    Physically factorized gate_proj module that stores Tucker core, factors,
    and uncompressed superweights, eliminating redundant parameter memory on GPU.
    """
    def __init__(self, W_orig, super_indices, active_coords, core, factors):
        super().__init__()
        self.num_coords, self.d_in = W_orig.shape
        self.register_buffer("super_indices", torch.tensor(super_indices, dtype=torch.long))
        self.register_buffer("active_coords", torch.tensor(active_coords, dtype=torch.long))
        
        all_special = set(super_indices).union(set(active_coords))
        remaining_indices = [i for i in range(self.num_coords) if i not in all_special]
        self.register_buffer("remaining_indices", torch.tensor(remaining_indices, dtype=torch.long))
        
        # Parameters physically stored on GPU
        self.superweights = torch.nn.Parameter(W_orig[super_indices, :].clone(), requires_grad=False)
        self.remaining_weights = torch.nn.Parameter(W_orig[remaining_indices, :].clone(), requires_grad=False)
        self.core = torch.nn.Parameter(core.clone(), requires_grad=False)
        self.factors = torch.nn.ParameterList([
            torch.nn.Parameter(f.clone(), requires_grad=False) for f in factors
        ])
        
    def forward(self, x):
        orig_shape = x.shape
        x_2d = x.reshape(-1, self.d_in)
        out = torch.empty((x_2d.shape[0], self.num_coords), device=x.device, dtype=x.dtype)
        
        # 1. Superweights
        out[:, self.super_indices] = (x_2d @ self.superweights.T).to(dtype=out.dtype)
        
        # 2. Remaining weights
        out[:, self.remaining_indices] = (x_2d @ self.remaining_weights.T).to(dtype=out.dtype)
        
        # 3. Tucker Active coordinates
        W_active = tucker_to_tensor((self.core, list(self.factors))).reshape(-1, self.d_in)
        out[:, self.active_coords] = (x_2d @ W_active.T).to(dtype=out.dtype)
        
        return out.reshape(*orig_shape[:-1], self.num_coords)

# Branch 1 Settings & Tucker Ranks (Tier 1 Ultra-Cut: [2, 20, 20] or Tier 2: [3, 80, 200])
NUM_CLUSTERS = 6
COORDS_PER_CLUSTER = 400
TARGET_TOTAL_ACTIVE = NUM_CLUSTERS * COORDS_PER_CLUSTER
TUCKER_RANKS = [2, 20, 20]  # Ultra-high compression tier for maximum VRAM reduction

print(f"Applying Branch 1 Factorization across all {NUM_LAYERS} layers (Ranks {TUCKER_RANKS})...")

for l in range(NUM_LAYERS):
    acts_l = acts_matrices[l]
    W_gate_l = W_gate_orig_all[l]
    
    # 1. Outlier Filter
    max_mags = np.max(np.abs(acts_l), axis=0)
    variances = np.var(acts_l, axis=0)
    super_mask = (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]
    
    inactive_mask = (np.mean(np.abs(acts_l) < 0.05, axis=0) > 0.90) & (~super_mask)
    normal_active_indices = np.where((~super_mask) & (~inactive_mask))[0]
    
    # 2. 1D Top Frequent Values
    rounded_acts = np.round(acts_l[:, normal_active_indices], decimals=1)
    top_vals = [np.unique(rounded_acts[:, i], return_counts=True)[0][np.argmax(np.unique(rounded_acts[:, i], return_counts=True)[1])]
                for i in range(len(normal_active_indices))]
    
    sorted_order = np.argsort(top_vals)[:TARGET_TOTAL_ACTIVE]
    selected_coords = normal_active_indices[sorted_order]
    
    cluster_slices = [
        W_gate_l[selected_coords[k*COORDS_PER_CLUSTER : (k+1)*COORDS_PER_CLUSTER], :].float().cpu()
        for k in range(NUM_CLUSTERS)
    ]
    T_l = torch.stack(cluster_slices, dim=0)
    
    # Tucker Decomposition
    core, factors = tucker(T_l, rank=TUCKER_RANKS, init='svd')
    
    # Structural Module Replacement on GPU
    factorized_module = TuckerFactorizedGateProj(
        W_orig=W_gate_l,
        super_indices=super_indices,
        active_coords=selected_coords,
        core=core,
        factors=factors
    ).to(device=model.device, dtype=model.dtype)
    
    model.model.layers[l].mlp.gate_proj = factorized_module

clear_cuda_cache()
print("All 26 gate_proj modules physically replaced with TuckerFactorizedGateProj.")

## Step 3: Measure Processed Model VRAM & Peak Inference Memory

We measure the physical VRAM occupied by the model with all 26 layers factorized:
1. **Processed Static Parameter VRAM**: Physical byte count of parameters on GPU.
2. **Processed Peak Runtime Memory**: Maximum VRAM reached during GLUE MNLI inference passes.
3. Verify accuracy retention on GLUE MNLI.

In [ ]:
# =====================================================================
# STEP 7: Processed Model Benchmark & Peak VRAM Measurement
# =====================================================================
clear_cuda_cache()

processed_param_vram = get_model_param_vram_mb(model)
processed_snapshot_before = get_cuda_memory_snapshot()

processed_gate_vram = sum(
    sum(p.numel() * p.element_size() for p in model.model.layers[l].mlp.gate_proj.parameters() if p.is_cuda)
    for l in range(NUM_LAYERS)
) / (1024 ** 2)

comp_preds = []
comp_gts = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Processed Model (Branch 1) Inference"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)
        next_token_logits = outputs.logits[0, -1, :]
        pred_label = torch.argmax(next_token_logits[label_token_ids]).item()
        comp_preds.append(pred_label)
        comp_gts.append(sample["label"])

processed_snapshot_after = get_cuda_memory_snapshot()
processed_accuracy = accuracy_score(comp_gts, comp_preds)
processed_peak_vram = processed_snapshot_after["max_allocated_mb"]
delta_acc = processed_accuracy - baseline_accuracy

print(f"\nProcessed Model Benchmark Results:")
print(f"  Accuracy:                {processed_accuracy * 100:.2f}% (Δ vs Baseline: {delta_acc * 100:+.2f}%)")
print(f"  Static Parameter VRAM:   {processed_param_vram:.2f} MB")
print(f"  Allocated VRAM:          {processed_snapshot_after['allocated_mb']:.2f} MB")
print(f"  Peak Runtime VRAM:       {processed_peak_vram:.2f} MB")

## Step 4: Comparative VRAM Synthesis, Savings Table & Global Time Log

We tabulate the physical VRAM savings, memory reduction percentages, and accuracy retention.

In [ ]:
# =====================================================================
# STEP 8: Comparative VRAM Synthesis & Artifact Export
# =====================================================================
gate_vram_saved = unprocessed_gate_vram - processed_gate_vram
gate_pct_saved = (gate_vram_saved / unprocessed_gate_vram) * 100.0

total_param_vram_saved = unprocessed_param_vram - processed_param_vram
total_pct_saved = (total_param_vram_saved / unprocessed_param_vram) * 100.0

peak_vram_diff = unprocessed_peak_vram - processed_peak_vram

print("=" * 95)
print(f"{'VRAM Metric':<40} | {'Unprocessed':<15} | {'Processed (Branch 1)':<20} | {'Reduction':<12}")
print("=" * 95)
print(f"{'26 gate_proj Weight VRAM (MB)':<40} | {unprocessed_gate_vram:>11.2f} MB | {processed_gate_vram:>16.2f} MB | {gate_vram_saved:>6.2f} MB ({gate_pct_saved:.1f}%)")
print(f"{'Total Model Parameter VRAM (MB)':<40} | {unprocessed_param_vram:>11.2f} MB | {processed_param_vram:>16.2f} MB | {total_param_vram_saved:>6.2f} MB ({total_pct_saved:.1f}%)")
print(f"{'Peak Runtime VRAM on MNLI (MB)':<40} | {unprocessed_peak_vram:>11.2f} MB | {processed_peak_vram:>16.2f} MB | {peak_vram_diff:>6.2f} MB")
print(f"{'GLUE MNLI Benchmark Accuracy':<40} | {baseline_accuracy * 100:>11.2f}% | {processed_accuracy * 100:>16.2f}% | {delta_acc * 100:>+6.2f}%")
print("=" * 95)

# Global Notebook Timing
total_notebook_runtime = time.time() - GLOBAL_NOTEBOOK_START_TIME
print(f"cummulative_time: {total_notebook_runtime:.2f}s")

# Save export artifact
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
output_path = artifact_dir / "vram_profiling_branch1_results.json"

export_data = {
    "experiment": "01_vram_profiling_branch1",
    "model_id": model_id,
    "num_layers": NUM_LAYERS,
    "ranks": TUCKER_RANKS,
    "unprocessed": {
        "gate_proj_vram_mb": round(unprocessed_gate_vram, 2),
        "total_model_param_vram_mb": round(unprocessed_param_vram, 2),
        "peak_runtime_vram_mb": round(unprocessed_peak_vram, 2),
        "accuracy": baseline_accuracy,
    },
    "processed_branch1": {
        "gate_proj_vram_mb": round(processed_gate_vram, 2),
        "total_model_param_vram_mb": round(processed_param_vram, 2),
        "peak_runtime_vram_mb": round(processed_peak_vram, 2),
        "accuracy": processed_accuracy,
    },
    "savings": {
        "gate_vram_saved_mb": round(gate_vram_saved, 2),
        "gate_vram_cut_pct": round(gate_pct_saved, 2),
        "total_param_vram_saved_mb": round(total_param_vram_saved, 2),
        "accuracy_delta": delta_acc,
    },
    "timing_summary": {
        "cummulative_time_sec": round(total_notebook_runtime, 3),
        "cell_timings": NOTEBOOK_TIMINGS,
    }
}

save_json_metrics(export_data, output_path)
print(f"\nVRAM profiling results saved to: {output_path}")